In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.normalisation import normalise_text, normalise_test_data
from src.lid import language_predictor
from src.bpe import BPETokenLearner

In [2]:
# import training data
with open("../data/train.en.txt") as f:
    eng_text = f.read()
eng_sentences = normalise_text(eng_text)

with open("../data/train.af.txt") as f:
    af_text = f.read()
af_sentences = normalise_text(af_text)

with open("../data/train.nl.txt") as f:
    nl_text = f.read()
nl_sentences = normalise_text(nl_text)  

with open("../data/train.xh.txt") as f:
    xh_text = f.read()
xh_sentences = normalise_text(xh_text)

with open("../data/train.zu.txt") as f:
    zu_text = f.read()
zu_sentences = normalise_text(zu_text)


# import validation data
with open("../data/val.en.txt") as f:
    eng_val_text = f.read()
eng_val_sentences = normalise_text(eng_val_text)

with open("../data/val.af.txt") as f:
    af_val_text = f.read()
af_val_sentences = normalise_text(af_val_text)

with open("../data/val.nl.txt") as f:
    nl_val_text = f.read()
nl_val_sentences = normalise_text(nl_val_text)

with open("../data/val.xh.txt") as f:
    xh_val_text = f.read()
xh_val_sentences = normalise_text(xh_val_text)

with open("../data/val.zu.txt") as f:
    zu_val_text = f.read()
zu_val_sentences = normalise_text(zu_val_text)

## Train the BPE token learners

In [3]:
bpe_eng = BPETokenLearner(num_merges=100)
bpe_af  = BPETokenLearner(num_merges=100)
bpe_nl  = BPETokenLearner(num_merges=100)
bpe_xh  = BPETokenLearner(num_merges=100)
bpe_zu  = BPETokenLearner(num_merges=100)

bpe_eng.fit(eng_sentences)
bpe_af.fit(af_sentences)
bpe_nl.fit(nl_sentences)
bpe_xh.fit(xh_sentences)
bpe_zu.fit(zu_sentences)

## Report first ten merges

In [4]:
models = {
    "eng": bpe_eng,
    "af": bpe_af,
    "nl": bpe_nl,
    "xh": bpe_xh,
    "zu": bpe_zu,
}

for lang, model in models.items():
    print(f"\n{lang.upper()} first 10 merges:")
    for i, pair in enumerate(model.get_merge_history()[:10], start=1):
        print(f"{i:2d}. {pair[0]!r} + {pair[1]!r} -> {pair[0] + pair[1]!r}")


ENG first 10 merges:
 1. 'e' + ' ' -> 'e '
 2. 's' + ' ' -> 's '
 3. 't' + 'h' -> 'th'
 4. 'd' + ' ' -> 'd '
 5. 'i' + 'n' -> 'in'
 6. 'a' + 'n' -> 'an'
 7. 'e' + 'r' -> 'er'
 8. 'o' + 'n' -> 'on'
 9. 'th' + 'e ' -> 'the '
10. 't' + ' ' -> 't '

AF first 10 merges:
 1. 'e' + ' ' -> 'e '
 2. 'n' + ' ' -> 'n '
 3. 'e' + 'r' -> 'er'
 4. 'd' + 'i' -> 'di'
 5. 'di' + 'e ' -> 'die '
 6. 's' + ' ' -> 's '
 7. 't' + ' ' -> 't '
 8. 'a' + 'n' -> 'an'
 9. 'e' + 'l' -> 'el'
10. 'e' + 'n' -> 'en'

NL first 10 merges:
 1. 'n' + ' ' -> 'n '
 2. 'e' + ' ' -> 'e '
 3. 'e' + 'n ' -> 'en '
 4. 'e' + 'r' -> 'er'
 5. 't' + ' ' -> 't '
 6. 'd' + 'e ' -> 'de '
 7. 'e' + 'n' -> 'en'
 8. 'a' + 'a' -> 'aa'
 9. 's' + ' ' -> 's '
10. 'e' + 'l' -> 'el'

XH first 10 merges:
 1. 'a' + ' ' -> 'a '
 2. 'e' + ' ' -> 'e '
 3. 'i' + ' ' -> 'i '
 4. 'o' + ' ' -> 'o '
 5. 'a' + 'n' -> 'an'
 6. 'k' + 'u' -> 'ku'
 7. 'n' + 'g' -> 'ng'
 8. 'e' + 'l' -> 'el'
 9. 'i' + 'n' -> 'in'
10. '^' + '^' -> '^^'

ZU first 10 merges:
 1

## Pairwise Subword Vocab Overlap

In [5]:
vocabs = {
    "eng": bpe_eng.get_vocab(),
    "af": bpe_af.get_vocab(),
    "nl": bpe_nl.get_vocab(),
    "xh": bpe_xh.get_vocab(),
    "zu": bpe_zu.get_vocab(),
}
langs = list(vocabs.keys())

print("\nPairwise BPE vocabulary overlap:")
for i in range(len(langs)):
    for j in range(i + 1, len(langs)):
        l1, l2 = langs[i], langs[j]
        overlap = vocabs[l1] & vocabs[l2]
        print(f"{l1}-{l2}: {len(overlap)} shared subwords")


Pairwise BPE vocabulary overlap:
eng-af: 99 shared subwords
eng-nl: 97 shared subwords
eng-xh: 75 shared subwords
eng-zu: 83 shared subwords
af-nl: 127 shared subwords
af-xh: 77 shared subwords
af-zu: 77 shared subwords
nl-xh: 66 shared subwords
nl-zu: 73 shared subwords
xh-zu: 124 shared subwords
